> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Joint fine-mapping of overlapping factors — VIGOR 2.12.1

Pass multiple selected traits/factors to one multivariate model. The native
Bayesian fit assigns inclusion to a shared variant with a vector of effects;
trait-specific directions are retained. This is not a colocalization posterior.
Peak overlap defines a candidate window, not evidence that both traits share a
causal SNP. The regional SNP set and LD must match all selected traits.

`trait_subset` selects responses and matching labelled G/E covariance entries.
Named animal rows align before fitting. Existing fitters use complete response
rows; missingness in excluded traits does not discard animals.

In [ ]:
import numpy as np
import pandas as pd
from vigor import fine_map_region
rng=np.random.default_rng(304);n=120;p=12
X=rng.binomial(2,.35,(n,p)).astype(float);X[:,1]=X[:,0]
B=np.zeros((p,3));B[4,1:]=[1.5,-1.2];B[8,0]=1.
Y=X@B+rng.normal(0,.3,(n,3))
ids=[f"i{i}" for i in range(n)];snps=[f"1:{100*i+1}" for i in range(p)]
X=pd.DataFrame(X,index=ids,columns=snps)
factors=pd.DataFrame(Y,index=ids,columns=["factor1","factor3","factor4"])
G=pd.DataFrame(np.diag([.2,.8,.7]),index=factors.columns,columns=factors.columns)
E=pd.DataFrame(np.eye(3)*.2,index=factors.columns,columns=factors.columns)
result=fine_map_region(X,factors.sample(frac=1,random_state=6),
    trait_subset=["factor4","factor3"],G=G,E=E,methods=("bayes",),
    L=1,hdbscan=False,compute_linear_shap=False)
assert result.metadata["selected_traits"]==["factor4","factor3"]
assert result.ranking.bayes_pip.idxmax()==snps[4]
print(result.ranking[["bayes_pip","ld_group"]].head())

In [ ]:
effects=result.fits["bayes"]["coef"]
assert effects.columns.tolist()==["factor4","factor3"]
assert effects.loc[snps[4],"factor4"]<0 and effects.loc[snps[4],"factor3"]>0
print(effects.loc[snps[4]])
print(result.metadata["n_complete_samples"], "complete animals")
print(result.fits["bayes"]["credible_sets"])


For a graph window, join `haplotype_graph_variants(graph)` by variant ID to
these rankings for LD/node/gene context. Use `haplotype_graph_features(graph)`
for an expert's additional haplotype inputs. The graph does not turn a joint
PIP into a conditional haplotype-effect estimate automatically.

A lead's default 6 Mb window is ±3 Mb. Select factors 3 and 4 together when
both have evidence in that region, or fit either separately by changing
`trait_subset`. Keep factor loading provenance and transform G/E into the
factor-score basis before passing them. LOCO sample whitening is distinct from
this trait-covariance transformation; supply consistently corrected locus
responses/genotypes if population structure requires it.

The PheWAS plotting helper groups nearby lead positions into regions. Such
physical grouping can choose the responses/window for a joint fit, but does
not itself fit a multivariate fine-mapping model.